In [1]:
import sqlite3
import pandas as pd
import logging
from datetime import date, datetime

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("pipeline.log"),   
    ],
    force=True
)
log = logging.getLogger(__name__)

In [3]:
sdm = sqlite3.connect("test.sqlite")
dwh = sqlite3.connect("sterschemav2.db")
log.info("Verbinding met SDM en DWH geopend")

In [4]:
cur = dwh.cursor()
cur_sdm = sdm.cursor()

In [5]:
# ETL datums
def etl_datums():
    log.info("START ETL: Datum_dim")

    dates = pd.read_sql("""
        SELECT datum FROM Fiets_Verkoop
        UNION
        SELECT datum FROM Accessoire_Verkoop
        UNION
        SELECT datum FROM Onderhoud
    """, sdm)
    log.info(f"  Verkoop/onderhoud datums geladen: {len(dates)} rijen")

    dates["jaar"]     = pd.to_datetime(dates["datum"]).dt.year
    dates["maand"]    = pd.to_datetime(dates["datum"]).dt.month
    dates["kwartaal"] = pd.to_datetime(dates["datum"]).dt.quarter
    dates["dag"]      = pd.to_datetime(dates["datum"]).dt.day

    inkoop_dates = pd.read_sql("""
        SELECT inkoopmaand AS maand, inkoopjaar AS jaar FROM Fiets_Inkoop
        UNION
        SELECT inkoopmaand AS maand, inkoopjaar AS jaar FROM Accessoire_Inkoop
    """, sdm)
    log.info(f"  Inkoop datums geladen: {len(inkoop_dates)} rijen")

    inkoop_dates["datum"] = inkoop_dates.apply(
        lambda r: f"{int(r['jaar'])}-{int(r['maand']):02d}-01", axis=1
    )
    inkoop_dates["dag"] = 1
    inkoop_dates["kwartaal"] = ((inkoop_dates["maand"] - 1) // 3 + 1).astype(int)

    all_dates = pd.concat([dates, inkoop_dates], ignore_index=True)
    all_dates = all_dates.drop_duplicates(subset=["datum"])
    log.info(f"  Unieke datums na deduplicatie: {len(all_dates)}")

    inserted = 0
    for _, rij in all_dates.iterrows():
        cur.execute("""
            INSERT OR IGNORE INTO Datum_dim (datum, jaar, maand, kwartaal, dag)
            VALUES (?, ?, ?, ?, ?)
        """, [rij["datum"], int(rij["jaar"]), int(rij["maand"]), int(rij["kwartaal"]), int(rij["dag"])])
        inserted += cur.rowcount

    dwh.commit()
    log.info(f"  Ingevoegd in Datum_dim: {inserted} rijen")
    log.info("KLAAR ETL: Datum_dim")

In [6]:
def etl_monteur():
    log.info("START ETL: Monteur_dim")

    monteur = pd.read_sql("""
        SELECT * FROM Accesoireverkoop_Monteur
        UNION
        SELECT * FROM Fietsverkoop_Monteur
        UNION
        SELECT * FROM Onderhoud_Monteur
    """, sdm)

    filiaal = pd.read_sql("""
        SELECT * FROM Accesoireverkoop_Filiaal
        UNION
        SELECT * FROM Fietsverkoop_Filiaal
        UNION
        SELECT * FROM Onderhoud_Filiaal
    """, sdm)

    merged = pd.merge(monteur, filiaal, left_on='filiaal', right_on='filiaalnr', suffixes=('_monteur', '_filiaal'))
    log.info(f"  Source rows after merge: {len(merged)}")

    monteur_huidig = pd.read_sql("SELECT * FROM Monteur_dim WHERE is_current = 1", dwh)

    tracked_columns = ['naam', 'woonplaats', 'uurloon', 'filiaalnaam', 'filiaaladres', 'filiaalprovincie']

    today = date.today().isoformat()
    inserted = 0
    expired = 0

    for _, nieuw in merged.iterrows():
        waarden = {
            'monteurnr':        nieuw['monteurnr'],
            'naam':             nieuw['naam_monteur'],
            'woonplaats':       nieuw['woonplaats'],
            'uurloon':          nieuw['uurloon'],
            'filiaalnaam':      nieuw['naam_filiaal'],
            'filiaaladres':     nieuw['adres'],
            'filiaalprovincie': nieuw['provincie']
        }

        bestaand = monteur_huidig[monteur_huidig['monteurnr'] == waarden['monteurnr']]

        if bestaand.empty:
            cur.execute("""
                INSERT INTO Monteur_dim 
                    (monteurnr, naam, woonplaats, uurloon, filiaalnaam, filiaaladres, filiaalprovincie,
                    effective_from, effective_to, is_current)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
            """, (waarden['monteurnr'], waarden['naam'], waarden['woonplaats'], waarden['uurloon'],
                waarden['filiaalnaam'], waarden['filiaaladres'], waarden['filiaalprovincie'], today))
            inserted += 1

        else:
            oud = bestaand.iloc[0]
            changed = any(waarden[col] != oud[col] for col in tracked_columns)

            if changed:
                cur.execute("""
                    UPDATE Monteur_dim
                    SET effective_to = ?, is_current = 0
                    WHERE monteurnr = ? AND is_current = 1
                """, (today, waarden['monteurnr']))
                expired += 1

                cur.execute("""
                    INSERT INTO Monteur_dim
                        (monteurnr, naam, woonplaats, uurloon, filiaalnaam, filiaaladres, filiaalprovincie,
                        effective_from, effective_to, is_current)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, NULL, 1)
                """, (waarden['monteurnr'], waarden['naam'], waarden['woonplaats'], waarden['uurloon'],
                    waarden['filiaalnaam'], waarden['filiaaladres'], waarden['filiaalprovincie'], today))
                inserted += 1

    dwh.commit()
    log.info(f"  Inserted: {inserted} rows | Expired: {expired} rows")
    log.info("DONE ETL: Monteur_dim")

In [7]:
def etl_klant():
    log.info("START ETL: Klant_dim")

    klant_nieuw = pd.read_sql("""
        SELECT * FROM Accesoireverkoop_Klant
        UNION
        SELECT * FROM Fietsverkoop_Klant
    """, sdm)
    log.info(f"  Customers from source: {len(klant_nieuw)} rows")

    klant_huidig = pd.read_sql("""
        SELECT * FROM Klant_dim WHERE is_current = 1
    """, dwh)

    tracked_columns = ['naam', 'adres', 'woonplaats', 'geslacht']  

    today = date.today().isoformat()
    inserted = 0
    expired = 0

    for _, nieuw in klant_nieuw.iterrows():
        bestaand = klant_huidig[klant_huidig['klantnr'] == nieuw['klantnr']]

        if bestaand.empty:
            cur.execute("""
                INSERT INTO Klant_dim 
                    (klantnr, naam, adres, woonplaats, geslacht, geboortedatum,
                    effective_from, effective_to, is_current)
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL, 1)
            """, (
                nieuw['klantnr'], nieuw['naam'], nieuw['adres'], nieuw['woonplaats'],
                nieuw['geslacht'], nieuw['geboortedatum'], today
            ))
            inserted += 1

        else:
            oud = bestaand.iloc[0]
            changed = any(nieuw[col] != oud[col] for col in tracked_columns)

            if changed:
                cur.execute("""
                    UPDATE Klant_dim 
                    SET effective_to = ?, is_current = 0
                    WHERE klantnr = ? AND is_current = 1
                """, (today, nieuw['klantnr']))
                expired += 1

                cur.execute("""
                    INSERT INTO Klant_dim 
                        (klantnr, naam, adres, woonplaats, geslacht, geboortedatum,
                        effective_from, effective_to, is_current)
                    VALUES (?, ?, ?, ?, ?, ?, ?, NULL, 1)
                """, (
                    nieuw['klantnr'], nieuw['naam'], nieuw['adres'], nieuw['woonplaats'],
                    nieuw['geslacht'], nieuw['geboortedatum'], today
                ))
                inserted += 1

    dwh.commit()
    log.info(f"  Inserted: {inserted} rows | Expired: {expired} rows")
    log.info("DONE ETL: Klant_dim")

In [8]:
# ETL product
def etl_product():
    log.info("START ETL: Product_dim")

    fiets = pd.read_sql("SELECT * FROM Fietsverkoop_Fiets", sdm)
    fiets = fiets.rename(columns={'fietsnr': 'productnr'})
    fiets['type'] = 'fiets'
    fiets['productnr'] = 'fiets_' + fiets['productnr'].astype(str)
    fiets['leveranciernr'] = None
    fiets['leverancier_naam'] = None
    fiets['leverancier_adres'] = None
    fiets['leverancier_woonplaats'] = None

    fabrikant = pd.read_sql("SELECT * FROM Fietsverkoop_Fabrikant", sdm)
    fabrikant = fabrikant.rename(columns={
        'naam':  'fabrikant_naam',
        'adres': 'fabrikant_adres',
        'plaats': 'fabrikant_plaats'
    })
    fiets = pd.merge(fiets, fabrikant, left_on='fabrikant', right_on='fabrikantnr')
    log.info(f"  Fietsen na merge met fabrikant: {len(fiets)} rijen")

    accessoire = pd.read_sql("SELECT * FROM Accessoire_Inkoop_Accessoire", sdm)
    accessoire = accessoire.rename(columns={'accessoirenr': 'productnr'})
    accessoire['type'] = 'accessoire'
    accessoire['productnr'] = 'accessoire_' + accessoire['productnr'].astype(str)
    accessoire['fabrikantnr'] = None
    accessoire['fabrikant_naam'] = None
    accessoire['fabrikant_adres'] = None
    accessoire['fabrikant_plaats'] = None
    accessoire['merk'] = None
    accessoire['kleur'] = None
    log.info(f"  Accessoires geladen: {len(accessoire)} rijen")

    leverancier = pd.read_sql("SELECT * FROM Accesoireverkoop_Leverancier", sdm)
    leverancier = leverancier.rename(columns={
        'naam':       'leverancier_naam',
        'adres':      'leverancier_adres',
        'woonplaats': 'leverancier_woonplaats'
    })
    accessoire = pd.merge(accessoire, leverancier, left_on='leverancier', right_on='leveranciernr')
    log.info(f"  Accessoires na merge met leverancier: {len(accessoire)} rijen")

    product = pd.concat([fiets, accessoire], sort=False)
    product = product.where(pd.notna(product), None)
    log.info(f"  Totaal producten na concat: {len(product)} rijen")

    inserted = 0
    for _, rij in product.iterrows():
        cur.execute("""
            INSERT INTO Product_dim (
                productnr, naam, type, soort, merk, kleur, standaardprijs, inkoopprijs,
                leveranciernr, leverancier_naam, leverancier_adres, leverancier_woonplaats,
                fabrikantnr, fabrikant_naam, fabrikant_adres, fabrikant_plaats
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(productnr) DO UPDATE SET
                naam                   = excluded.naam,
                type                   = excluded.type,
                soort                  = excluded.soort,
                merk                   = excluded.merk,
                kleur                  = excluded.kleur,
                standaardprijs         = excluded.standaardprijs,
                inkoopprijs            = excluded.inkoopprijs,
                leveranciernr          = excluded.leveranciernr,
                leverancier_naam       = excluded.leverancier_naam,
                leverancier_adres      = excluded.leverancier_adres,
                leverancier_woonplaats = excluded.leverancier_woonplaats,
                fabrikantnr            = excluded.fabrikantnr,
                fabrikant_naam         = excluded.fabrikant_naam,
                fabrikant_adres        = excluded.fabrikant_adres,
                fabrikant_plaats       = excluded.fabrikant_plaats
        """, (
            rij['productnr'], rij.get('naam'), rij['type'], rij['soort'], rij.get('merk'), rij.get('kleur'),
            rij['standaardprijs'], rij['inkoopprijs'],
            rij.get('leveranciernr'), rij.get('leverancier_naam'), rij.get('leverancier_adres'), rij.get('leverancier_woonplaats'),
            rij.get('fabrikantnr'), rij.get('fabrikant_naam'), rij.get('fabrikant_adres'), rij.get('fabrikant_plaats')
        ))
        inserted += cur.rowcount

    dwh.commit()
    log.info(f"  Ingevoegd in Product_dim: {inserted} rijen")
    log.info("KLAAR ETL: Product_dim")

In [9]:
def get_key(cur, table, nr_col, nr_val, scd2=False):
    if scd2:
        cur.execute(f"SELECT rowid FROM {table} WHERE {nr_col} = ? AND is_current = 1", (nr_val,))
    else:
        cur.execute(f"SELECT rowid FROM {table} WHERE {nr_col} = ?", (nr_val,))
    row = cur.fetchone()
    if row is None:
        log.warning(f"  Geen key gevonden in {table} voor {nr_col}={nr_val!r}")
    return row[0] if row else None

In [10]:
# ETL Fact_Verkoop
def etl_verkoop():
    log.info("START ETL: Fact_Verkoop")

    fiets_vk = pd.read_sql("SELECT *, 'fiets' as prodtype FROM Fiets_Verkoop", sdm)
    fiets_vk = fiets_vk.rename(columns={'fiets': 'productnr'})

    acc_vk = pd.read_sql("SELECT *, 'accessoire' as prodtype FROM Accessoire_Verkoop", sdm)
    acc_vk = acc_vk.rename(columns={'accessoire': 'productnr'})

    verkoop = pd.concat([fiets_vk, acc_vk], sort=False)
    verkoop = verkoop.where(pd.notna(verkoop), None)
    log.info(f"  Verkoopregels geladen: {len(verkoop)} rijen")

    inserted = 0
    for _, rij in verkoop.iterrows():
        datum_key   = get_key(cur, 'Datum_dim',   'datum',     rij['datum'])
        klant_key   = get_key(cur, 'Klant_dim',   'klantnr',   rij['klant'], scd2=True)
        product_key = get_key(cur, 'Product_dim', 'productnr', rij['prodtype'] + '_' + str(rij['productnr']))
        monteur_key = get_key(cur, 'Monteur_dim', 'monteurnr', rij['monteur'], scd2=True)

        aantal       = rij['aantal']
        verkoopprijs = round(float(rij['verkoopprijs']), 2)

        cur.execute("SELECT inkoopprijs FROM Product_dim WHERE rowid = ?", (product_key,))
        inkoopprijs = round(float(cur.fetchone()[0]) if product_key else 0, 2)

        totaleprijs       = round(aantal * verkoopprijs, 2)
        winst_per_product = round(verkoopprijs - inkoopprijs, 2)
        totalewinst       = round(aantal * winst_per_product, 2)

        cur.execute("""
            INSERT OR IGNORE INTO Fact_Verkoop (
                datum_key, klant_key, product_key, monteur_key,
                aantal, verkoopprijs, totaleprijs, winst_per_product, totalewinst
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (datum_key, klant_key, product_key, monteur_key,
            aantal, verkoopprijs, totaleprijs, winst_per_product, totalewinst))
        inserted += cur.rowcount

    dwh.commit()
    log.info(f"  Ingevoegd in Fact_Verkoop: {inserted} rijen")
    log.info("KLAAR ETL: Fact_Verkoop")

In [11]:
# ETL Fact_Onderhoud
def etl_onderhoud():
    log.info("START ETL: Fact_Onderhoud")

    onderhoud = pd.read_sql("SELECT * FROM Onderhoud", sdm)
    log.info(f"  Onderhoud geladen: {len(onderhoud)} rijen")

    fmt = "%H:%M:%S.%f"

    inserted = 0
    for _, rij in onderhoud.iterrows():
        datum_key   = get_key(cur, 'Datum_dim',   'datum',     rij['datum'])
        product_key = get_key(cur, 'Product_dim', 'productnr', 'fiets_' + str(rij['fiets']))
        monteur_key = get_key(cur, 'Monteur_dim', 'monteurnr', rij['monteur'], scd2=True)

        start = datetime.strptime(rij['starttijd'][:-3], fmt)
        eind  = datetime.strptime(rij['eindtijd'][:-3],  fmt)
        duur  = int((eind - start).seconds / 60)

        cur.execute("""
            INSERT OR IGNORE INTO Fact_Onderhoud (datum_key, product_key, monteur_key, duur)
            VALUES (?, ?, ?, ?)
        """, (datum_key, product_key, monteur_key, duur))
        inserted += cur.rowcount

    dwh.commit()
    log.info(f"  Ingevoegd in Fact_Onderhoud: {inserted} rijen")
    log.info("KLAAR ETL: Fact_Onderhoud")

In [12]:
# ETL Fact_Inkoop
def etl_inkoop():
    log.info("START ETL: Fact_Inkoop")

    fiets_ink = pd.read_sql("SELECT * FROM Fiets_Inkoop", sdm)
    fiets_ink = fiets_ink.rename(columns={'fiets': 'productnr'})
    fiets_ink['prodtype'] = 'fiets'

    acc_ink = pd.read_sql("SELECT * FROM Accessoire_Inkoop", sdm)
    acc_ink = acc_ink.rename(columns={'accessoire': 'productnr'})
    acc_ink['prodtype'] = 'accessoire'

    inkoop = pd.concat([fiets_ink, acc_ink], sort=False)
    inkoop = inkoop.where(pd.notna(inkoop), None)
    log.info(f"  Inkoop geladen: {len(inkoop)} rijen")

    inserted = 0
    for _, rij in inkoop.iterrows():
        datum_str   = f"{int(rij['inkoopjaar'])}-{int(rij['inkoopmaand']):02d}-01"
        datum_key   = get_key(cur, 'Datum_dim',   'datum',     datum_str)
        product_key = get_key(cur, 'Product_dim', 'productnr', rij['prodtype'] + '_' + str(rij['productnr']))

        cur.execute("""
            INSERT OR IGNORE INTO Fact_Inkoop (datum_key, product_key, aantal)
            VALUES (?, ?, ?)
        """, (datum_key, product_key, int(rij['aantal'])))
        inserted += cur.rowcount

    dwh.commit()
    log.info(f"  Ingevoegd in Fact_Inkoop: {inserted} rijen")
    log.info("KLAAR ETL: Fact_Inkoop")

In [13]:
def empty_tables():    
    log.info("START: Alle tabellen legen")
    cur.executescript("""
        DELETE FROM Fact_Inkoop;
        DELETE FROM Fact_Verkoop;
        DELETE FROM Fact_Onderhoud;
        DELETE FROM Datum_dim;
        DELETE FROM Klant_dim;
        DELETE FROM Monteur_dim;
        DELETE FROM Product_dim;
    """)
    dwh.commit()
    log.info("KLAAR: Alle tabellen geleegd")

In [14]:
empty_tables()

In [15]:
etl_datums()
etl_klant()
etl_monteur()
etl_product()
etl_onderhoud()
etl_verkoop()
etl_inkoop()

In [16]:
cur_sdm.executescript("""
    UPDATE Onderhoud_Monteur
    SET woonplaats = 'Amsterdam'
    WHERE monteurnr = 11;
                      
    INSERT INTO Onderhoud_Monteur (monteurnr, naam, woonplaats, uurloon, filiaal)
    VALUES (101, 'Jan Pietersen', 'Utrecht', 18.5, 1);

    INSERT INTO Onderhoud_Monteur (monteurnr, naam, woonplaats, uurloon, filiaal)
    VALUES (102, 'Lisa Visser', 'Rotterdam', 21.0, 2);
""")

pd.read_sql("""
    SELECT * FROM Onderhoud_Monteur
""", sdm)

,monteurnr,naam,woonplaats,uurloon,filiaal
0,1,Tom van Dijk,Amsterdam,19.50,1
1,2,Lisa Verhoeven,Amstelveen,18.75,1
2,3,Kevin Jaspers,Zaandam,20.00,1
3,4,Romy Willems,Haarlem,17.80,2
4,5,Daan de Groot,Rotterdam,21.00,3
5,6,Sophie de Leeuw,Schiedam,19.90,3
6,7,Mark van Vliet,Dordrecht,20.25,3
7,8,Jesse Peters,Leiden,16.95,4
8,9,Eva Meijer,Zoetermeer,18.40,4
9,10,Milan de Bruin,Zaandam,17.25,5


In [17]:
etl_monteur()

pd.read_sql("SELECT * FROM Monteur_dim", dwh)

,monteur_key,monteurnr,naam,woonplaats,uurloon,filiaalnaam,filiaaladres,filiaalprovincie,effective_from,effective_to,is_current
0,246,1,Tom van Dijk,Amsterdam,19.50,BikeWorld Amsterdam,Prinsengracht 100,Noord-Holland,2026-04-01,NaN,1
1,247,2,Lisa Verhoeven,Amstelveen,18.75,BikeWorld Amsterdam,Prinsengracht 100,Noord-Holland,2026-04-01,NaN,1
2,248,3,Kevin Jaspers,Zaandam,20.00,BikeWorld Amsterdam,Prinsengracht 100,Noord-Holland,2026-04-01,NaN,1
3,249,4,Romy Willems,Haarlem,17.80,FietsGigant Haarlem,Grote Markt 22,Noord-Holland,2026-04-01,NaN,1
4,250,5,Daan de Groot,Rotterdam,21.00,FietsExpress Rotterdam,Coolsingel 55,Zuid-Holland,2026-04-01,NaN,1
5,251,6,Sophie de Leeuw,Schiedam,19.90,FietsExpress Rotterdam,Coolsingel 55,Zuid-Holland,2026-04-01,NaN,1
6,252,7,Mark van Vliet,Dordrecht,20.25,FietsExpress Rotterdam,Coolsingel 55,Zuid-Holland,2026-04-01,NaN,1
7,253,8,Jesse Peters,Leiden,16.95,CycleCity Leiden,Breestraat 48,Zuid-Holland,2026-04-01,NaN,1
8,254,9,Eva Meijer,Zoetermeer,18.40,CycleCity Leiden,Breestraat 48,Zuid-Holland,2026-04-01,NaN,1
9,255,10,Milan de Bruin,Zaandam,17.25,FietsGigant Haarlem,Grote Markt 22,Noord-Holland,2026-04-01,2026-04-01,0


In [18]:
cur_sdm.executescript("""
    INSERT INTO Fiets_Verkoop (fiets_verkoopnr, datum, aantal, verkoopprijs, klant, fiets, monteur)
    VALUES (999, '2025-1-1', 5, 600.5, 12, 36, 11);
""")

pd.read_sql("SELECT * FROM Fiets_Verkoop WHERE Monteur = 11;", sdm)

,fiets_verkoopnr,datum,aantal,verkoopprijs,klant,fiets,monteur
0,999,2025-1-1,5,600.5,12,36,11


In [19]:
etl_verkoop()

pd.read_sql("SELECT * FROM Fact_Verkoop WHERE aantal = 5", dwh)

,id,datum_key,klant_key,product_key,monteur_key,aantal,verkoopprijs,totaleprijs,winst_per_product,totalewinst
0,401,None,12,1268,263,5,600.5,3002.5,-5.5,-27.5


In [89]:
cur_sdm.executescript("""
    UPDATE Onderhoud_Monteur
    SET woonplaats = 'Haarlem'
    WHERE monteurnr = 11;

    DELETE FROM Onderhoud_Monteur WHERE monteurnr IN (101, 102);
""")

cur.executescript("""
    DELETE FROM Monteur_dim WHERE monteurnr IN (101, 102);
    DELETE FROM Monteur_dim WHERE monteurnr = 11 AND is_current = 1;
    UPDATE Monteur_dim 
    SET effective_to = NULL, is_current = 1 
""")

In [90]:
cur_sdm.executescript("DELETE FROM Fiets_Verkoop WHERE fiets_verkoopnr = 999")

In [ ]:
cur.executescript("""
    DROP TABLE Fact_Verkoop;
                  
    CREATE TABLE Fact_Verkoop (
    id INTEGER PRIMARY KEY,
    datum_key INT,
    klant_key INT,
    product_key INT,
    monteur_key INT,
    aantal INT,
    verkoopprijs FLOAT,
    totaleprijs FLOAT,
    winst_per_product FLOAT,
    totalewinst FLOAT,
    FOREIGN KEY (datum_key) REFERENCES Datum_dim (datum_key),
    FOREIGN KEY (klant_key) REFERENCES Klant_dim (klant_key),
    FOREIGN KEY (product_key) REFERENCES Product_dim (product_key),
    FOREIGN KEY (monteur_key) REFERENCES Monteur_dim (monteur_key)
);
""")